# The Silent Gap — ASL Sign Language Challenge

**Final Kaggle Notebook — ConvNeXt Tiny Transfer Learning**

This Kaggle-ready notebook trains an image classification model for 29 ASL classes: **A–Z, space, del, nothing**.

It is designed to run inside Kaggle. Attach the competition data using:

`Right sidebar → Add Input → Competition Data`

The notebook automatically searches `/kaggle/input` for `train/`, `test/`, and `sample_submission.csv`, then saves outputs to `/kaggle/working/outputs_final_kaggle/`.

> ⚠️ **Before running:** Go to `Settings → Accelerator` and select **GPU T4 x2** or **P100**. Also enable **Internet** under `Settings → Internet` — it is required to download pretrained ConvNeXt weights from `timm`.

**Do not add `kaggle.json` or Google Drive cells in this Kaggle version.**

**Estimated runtime:** ~3–6 hours for `strong_3fold` (3 folds × 18 epochs) on a T4/P100 GPU (well within Kaggle's 9-hour limit).


In [1]:
# ============================================================
# 1) INSTALL REQUIRED LIBRARIES
# ============================================================
# Kaggle usually has most packages preinstalled. timm may need installation.

import importlib.util

if importlib.util.find_spec("timm") is None:
    !pip install -q timm
else:
    print("timm already installed")


timm already installed


In [2]:
# ============================================================
# 2) IMPORTS, SEED, DEVICE
# ============================================================

import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import timm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tqdm.auto import tqdm

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not available. Training will be very slow on CPU.")


Device: cuda
GPU: Tesla T4


In [3]:
# ============================================================
# 3) CONFIG
# ============================================================

CLASSES = [
    "A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M",
    "N", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z",
    "space", "del", "nothing"
]

CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASSES)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Choose one:
# quick        = 1 fold, fast sanity run
# strong_3fold = 3 folds, safer final run
RUN_MODE = "strong_3fold"

RUN_CONFIGS = {
    "quick": {
        "model_names": ["convnext_tiny.fb_in22k_ft_in1k"],
        "n_folds": 3,
        "folds_to_train": [0],
        "epochs": 5,
        "img_size": 224,
        "batch_size": 32,
        "lr": 2e-4,
        "weight_decay": 1e-4,
        "label_smoothing": 0.05,
    },
    "strong_3fold": {
        "model_names": ["convnext_tiny.fb_in22k_ft_in1k"],
        "n_folds": 3,
        "folds_to_train": [0, 1, 2],
        "epochs": 18,
        "img_size": 224,
        "batch_size": 32,
        "lr": 2e-4,
        "weight_decay": 1e-4,
        "label_smoothing": 0.05,
    },
}

cfg = RUN_CONFIGS[RUN_MODE]
MODEL_NAMES = cfg["model_names"]
N_FOLDS = cfg["n_folds"]
FOLDS_TO_TRAIN = cfg["folds_to_train"]
EPOCHS = cfg["epochs"]
IMG_SIZE = cfg["img_size"]
BATCH_SIZE = cfg["batch_size"]
LR = cfg["lr"]
WEIGHT_DECAY = cfg["weight_decay"]
LABEL_SMOOTHING = cfg["label_smoothing"]

NUM_WORKERS = 2
PIN_MEMORY = torch.cuda.is_available()

# Kaggle paths
INPUT_ROOT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
OUT_DIR = WORKING_DIR / "outputs_final_kaggle"
MODEL_DIR = OUT_DIR / "models"
SUBMISSION_DIR = OUT_DIR / "submissions"
LOG_DIR = OUT_DIR / "logs"
REPORT_DIR = OUT_DIR / "reports"
OOF_DIR = OUT_DIR / "oof_predictions"

for d in [OUT_DIR, MODEL_DIR, SUBMISSION_DIR, LOG_DIR, REPORT_DIR, OOF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("RUN_MODE:", RUN_MODE)
print("Models:", MODEL_NAMES)
print("Folds:", FOLDS_TO_TRAIN, "of", N_FOLDS)
print("Epochs:", EPOCHS)
print("Output directory:", OUT_DIR)


RUN_MODE: strong_3fold
Models: ['convnext_tiny.fb_in22k_ft_in1k']
Folds: [0, 1, 2] of 3
Epochs: 18
Output directory: /kaggle/working/outputs_final_kaggle


In [4]:
# ============================================================
# 4) AUTO-DETECT KAGGLE DATASET FOLDERS
# ============================================================

def count_images(folder):
    folder = Path(folder)
    return sum(1 for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXTS)

def find_dataset_dir(root=INPUT_ROOT):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Input root not found: {root}")

    candidates = []
    for p in [root] + [x for x in root.rglob("*") if x.is_dir()]:
        train = p / "train"
        test = p / "test"
        if train.exists() and test.exists():
            train_count = count_images(train)
            test_count = count_images(test)
            if train_count > 0 and test_count > 0:
                candidates.append((p, train_count, test_count))

    if not candidates:
        print("Available /kaggle/input folders:")
        for p in root.glob("*"):
            print(" -", p)
        raise FileNotFoundError("Could not find a dataset folder containing train/ and test/ with image files.")

    # choose candidate with largest train count
    candidates = sorted(candidates, key=lambda x: x[1], reverse=True)
    return candidates[0][0]

DATA_DIR = find_dataset_dir(INPUT_ROOT)
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

sample_candidates = list(DATA_DIR.rglob("sample_submission.csv")) + list(INPUT_ROOT.rglob("sample_submission.csv"))
SAMPLE_SUB_PATH = sample_candidates[0] if sample_candidates else None

print("DATA_DIR:", DATA_DIR)
print("TRAIN_DIR:", TRAIN_DIR, "images:", count_images(TRAIN_DIR))
print("TEST_DIR:", TEST_DIR, "images:", count_images(TEST_DIR))
print("SAMPLE_SUB_PATH:", SAMPLE_SUB_PATH)


DATA_DIR: /kaggle/input/competitions/sign-language-contest
TRAIN_DIR: /kaggle/input/competitions/sign-language-contest/train images: 69600
TEST_DIR: /kaggle/input/competitions/sign-language-contest/test images: 26000
SAMPLE_SUB_PATH: None


In [6]:
# ============================================================
# 5) BUILD TRAIN/TEST DATAFRAMES
# ============================================================

rows = []
unknown_folders = []

for cls_dir in sorted(TRAIN_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    label = cls_dir.name
    if label not in CLASS_TO_IDX:
        unknown_folders.append(label)
        continue
    for p in cls_dir.rglob("*"):
        if p.suffix.lower() in IMAGE_EXTS:
            rows.append({
                "image_path": str(p),
                "image_id": p.name,
                "label": label,
                "target": CLASS_TO_IDX[label],
            })

if unknown_folders:
    print("Warning: unknown class folders ignored:", unknown_folders)

train_df = pd.DataFrame(rows)
if train_df.empty:
    raise ValueError("No training images found.")

all_test_paths = sorted([p for p in TEST_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTS], key=lambda x: x.name)
test_path_map = {p.name: str(p) for p in all_test_paths}

if SAMPLE_SUB_PATH is not None:
    sample_df = pd.read_csv(SAMPLE_SUB_PATH)
    if "image_id" not in sample_df.columns:
        raise ValueError("sample_submission.csv must contain an image_id column.")
    test_df = sample_df[["image_id"]].copy()
    test_df["image_path"] = test_df["image_id"].map(test_path_map)
    missing = test_df["image_path"].isna().sum()
    if missing:
        missing_ids = test_df.loc[test_df["image_path"].isna(), "image_id"].head(10).tolist()
        raise FileNotFoundError(f"{missing} sample_submission image_ids not found in test folder. Examples: {missing_ids}")
else:
    sample_df = None
    test_df = pd.DataFrame({
        "image_id": [p.name for p in all_test_paths],
        "image_path": [str(p) for p in all_test_paths],
    })

print("Train images:", len(train_df))
print("Test images:", len(test_df))
print("Class count:", train_df["label"].nunique())
print("Class distribution:")
print(train_df["label"].value_counts().sort_index())

assert train_df["label"].nunique() == NUM_CLASSES, "Expected exactly 29 train classes."
assert len(test_df) > 0, "No test images found."


Train images: 69600
Test images: 26000
Class count: 29
Class distribution:
label
A          2400
B          2400
C          2400
D          2400
E          2400
F          2400
G          2400
H          2400
I          2400
J          2400
K          2400
L          2400
M          2400
N          2400
O          2400
P          2400
Q          2400
R          2400
S          2400
T          2400
U          2400
V          2400
W          2400
X          2400
Y          2400
Z          2400
del        2400
nothing    2400
space      2400
Name: count, dtype: int64


In [7]:
# ============================================================
# 6) STRATIFIED FOLDS
# ============================================================

train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
train_df["fold"] = -1

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df["target"])):
    train_df.loc[val_idx, "fold"] = fold

print(train_df["fold"].value_counts().sort_index())
print("Class count per fold:")
print(pd.crosstab(train_df["fold"], train_df["label"]))

train_df.to_csv(OUT_DIR / "train_folds.csv", index=False)
test_df.to_csv(OUT_DIR / "test_df.csv", index=False)
print("Saved:", OUT_DIR / "train_folds.csv")


fold
0    23200
1    23200
2    23200
Name: count, dtype: int64
Class count per fold:
label    A    B    C    D    E    F    G    H    I    J  ...    T    U    V  \
fold                                                     ...                  
0      800  800  800  800  800  800  800  800  800  800  ...  800  800  800   
1      800  800  800  800  800  800  800  800  800  800  ...  800  800  800   
2      800  800  800  800  800  800  800  800  800  800  ...  800  800  800   

label    W    X    Y    Z  del  nothing  space  
fold                                            
0      800  800  800  800  800      800    800  
1      800  800  800  800  800      800    800  
2      800  800  800  800  800      800    800  

[3 rows x 29 columns]
Saved: /kaggle/working/outputs_final_kaggle/train_folds.csv


In [9]:
# ============================================================
# 7) TRANSFORMS + DATASET
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_train_tf(img_size):
    return T.Compose([
        T.Resize((int(img_size * 1.15), int(img_size * 1.15))),
        T.RandomResizedCrop(img_size, scale=(0.80, 1.00), ratio=(0.90, 1.10)),
        T.RandomRotation(degrees=10),
        T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def get_eval_tf(img_size):
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

class SignDataset(Dataset):
    def __init__(self, df, transform=None, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        if self.has_labels:
            target = int(row["target"])
            return image, target
        return image


In [10]:
# ============================================================
# 8) MODEL + TRAINING HELPERS
# ============================================================

def safe_model_name(model_name):
    return model_name.replace("/", "_").replace(".", "_").replace("-", "_")

def create_model(model_name, num_classes=NUM_CLASSES):
    try:
        model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    except Exception as e:
        print("WARNING: pretrained=True failed. Falling back to pretrained=False.")
        print("Reason:", repr(e))
        model = timm.create_model(model_name, pretrained=False, num_classes=num_classes)
    return model

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="train", leave=False)
    for images, targets in pbar:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        preds = logits.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += batch_size
        pbar.set_postfix(loss=total_loss / max(total, 1), acc=correct / max(total, 1))

    return total_loss / total, correct / total

@torch.no_grad()
def validate_one_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, targets in tqdm(loader, desc="valid", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = criterion(logits, targets)

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        preds = logits.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += batch_size

    return total_loss / total, correct / total


In [13]:
# ============================================================
# 9) TRAIN MODEL / FOLDS
# ============================================================

all_history = []
trained_checkpoints = []

for model_name in MODEL_NAMES:
    model_tag = safe_model_name(model_name)

    for fold in FOLDS_TO_TRAIN:
        print("" + "=" * 80)
        print(f"Training model={model_name} | fold={fold}")
        print("=" * 80)

        train_part = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_part = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_ds = SignDataset(train_part, transform=get_train_tf(IMG_SIZE), has_labels=True)
        val_ds = SignDataset(val_part, transform=get_eval_tf(IMG_SIZE), has_labels=True)

        train_loader = DataLoader(
            train_ds, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False
        )
        val_loader = DataLoader(
            val_ds, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False
        )

        model = create_model(model_name, NUM_CLASSES).to(DEVICE)
        criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        scaler = torch.amp.GradScaler(device="cuda", enabled=torch.cuda.is_available())

        best_acc = -1.0
        best_loss = float("inf")
        ckpt_path = MODEL_DIR / f"{model_tag}_fold{fold}_best.pth"

        start_time = time.time()
        for epoch in range(1, EPOCHS + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
            val_loss, val_acc = validate_one_epoch(model, val_loader, criterion)
            scheduler.step()

            row = {
                "model_name": model_name,
                "fold": fold,
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "lr": optimizer.param_groups[0]["lr"],
            }
            all_history.append(row)
            pd.DataFrame(all_history).to_csv(OUT_DIR / "training_history.csv", index=False)

            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
                f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
            )

            if val_acc > best_acc or (val_acc == best_acc and val_loss < best_loss):
                best_acc = val_acc
                best_loss = val_loss
                torch.save({
                    "model_name": model_name,
                    "fold": fold,
                    "epoch": epoch,
                    "best_acc": best_acc,
                    "best_loss": best_loss,
                    "state_dict": model.state_dict(),
                    "classes": CLASSES,
                    "img_size": IMG_SIZE,
                    "run_mode": RUN_MODE,
                }, ckpt_path)
                print("Saved best checkpoint:", ckpt_path)

        elapsed = time.time() - start_time
        print(f"Fold {fold} complete. Best val_acc={best_acc:.6f}. Time={elapsed/60:.1f} min")
        trained_checkpoints.append(str(ckpt_path))

# Also recover any existing checkpoints in the output model folder
for p in sorted(MODEL_DIR.glob("*_best.pth")):
    if str(p) not in trained_checkpoints:
        trained_checkpoints.append(str(p))

pd.DataFrame({"checkpoint": trained_checkpoints}).to_csv(OUT_DIR / "trained_checkpoints.csv", index=False)
print("Training complete.")
print("Checkpoints:")
for c in trained_checkpoints:
    print(" -", c)


Training model=convnext_tiny.fb_in22k_ft_in1k | fold=0


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

train:   0%|          | 0/1450 [00:00<?, ?it/s]

valid:   0%|          | 0/725 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b41a759dee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    Exception ignored in: self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7b41a759dee0>

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
if w.is_alive():    
self._shutdown_workers() 
 Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 <function _MultiProcessingDataLoaderIter.__del__ at 0x7b41a759dee0>    if w.is_alive(): 
 
Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 

...

train:   0%|          | 0/1450 [00:00<?, ?it/s]

valid:   0%|          | 0/725 [00:00<?, ?it/s]

Epoch 01/18 | train_loss=0.4558 train_acc=0.9716 | val_loss=0.3568 val_acc=0.9996
Saved best checkpoint: /kaggle/working/outputs_final_kaggle/models/convnext_tiny_fb_in22k_ft_in1k_fold1_best.pth


train:   0%|          | 0/1450 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [14]:
# ============================================================
# 10) INFERENCE HELPERS
# ============================================================

import torch
import timm
import numpy as np
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Safety: define classes if not already defined
if "CLASSES" not in dir():
    CLASSES = [
        "A","B","C","D","E","F","G","H","I","J","K","L","M",
        "N","O","P","Q","R","S","T","U","V","W","X","Y","Z",
        "space","del","nothing"
    ]
NUM_CLASSES = len(CLASSES)
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASSES)}

# ── Model builder ─────────────────────────────────────────────────────────────
def build_model_for_checkpoint(model_name, num_classes=29):
    model = timm.create_model(
        model_name,
        pretrained=False,
        num_classes=num_classes
    )
    return model

# ── Load checkpoint list from disk ────────────────────────────────────────────
def load_checkpoint_list():
    csv_path = OUT_DIR / "trained_checkpoints.csv"
    if csv_path.exists():
        ckpts = pd.read_csv(csv_path)["checkpoint"].dropna().astype(str).tolist()
    else:
        ckpts = []
    # Also recover any .pth files directly from model folder
    for p in sorted(MODEL_DIR.glob("*_best.pth")):
        if str(p) not in ckpts:
            ckpts.append(str(p))
    if not ckpts:
        raise FileNotFoundError("No trained checkpoints found. Run the training cell first.")
    return ckpts

# ── Core inference function ───────────────────────────────────────────────────
@torch.no_grad()
def predict_checkpoint(ckpt_path, loader):
    ckpt_path = str(ckpt_path)
    print("\nLoading checkpoint:", ckpt_path)

    checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    # Standard notebook checkpoint format
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        model_name = checkpoint.get("model_name", "convnext_tiny.fb_in22k_ft_in1k")
        state_dict = checkpoint["state_dict"]
        print(f"  Model     : {model_name}")
        print(f"  Epoch     : {checkpoint.get('epoch', 'unknown')}")
        print(f"  Best acc  : {checkpoint.get('best_acc', 'unknown')}")
    else:
        # Fallback: raw state dict
        model_name = "convnext_tiny.fb_in22k_ft_in1k"
        state_dict = checkpoint
        print("  Using fallback model name:", model_name)

    model = build_model_for_checkpoint(model_name, NUM_CLASSES)
    model.load_state_dict(state_dict, strict=True)
    model.to(DEVICE)
    model.eval()

    all_probs = []
    for batch in tqdm(loader, desc=f"Predicting {Path(ckpt_path).name}", leave=False):
        if isinstance(batch, torch.Tensor):
            images = batch
        elif isinstance(batch, (list, tuple)):
            images = batch[0]
        else:
            raise TypeError(f"Unexpected batch type: {type(batch)}")

        images = images.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(images)
            probs  = torch.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    print(f"  Output shape: {all_probs.shape}")
    return all_probs

print("predict_checkpoint function is ready.")

Using device: cuda
predict_checkpoint function is ready.


In [15]:
# ============================================================
# 11) VALIDATION DIAGNOSTICS / OOF PREDICTIONS
# ============================================================

trained_checkpoints = load_checkpoint_list()

oof_parts = []
for ckpt_path in trained_checkpoints:
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    fold = int(ckpt.get("fold", -1))
    if fold < 0:
        print("Skipping OOF for checkpoint with unknown fold:", ckpt_path)
        continue

    val_part = train_df[train_df["fold"] == fold].reset_index(drop=True).copy()
    val_ds = SignDataset(val_part, transform=get_eval_tf(IMG_SIZE), has_labels=True)
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )

    probs = predict_checkpoint(ckpt_path, val_loader)
    preds = probs.argmax(axis=1)

    val_part["pred_target"] = preds
    val_part["pred_label"] = [IDX_TO_CLASS[i] for i in preds]
    val_part["correct"] = val_part["target"] == val_part["pred_target"]
    val_part["checkpoint"] = ckpt_path
    oof_parts.append(val_part)

if oof_parts:
    oof_df = pd.concat(oof_parts, ignore_index=True)
    oof_acc = accuracy_score(oof_df["label"], oof_df["pred_label"])
    print("OOF accuracy:", oof_acc)

    oof_df.to_csv(OUT_DIR / "oof_predictions_all.csv", index=False)

    cm = confusion_matrix(oof_df["label"], oof_df["pred_label"], labels=CLASSES)
    cm_df = pd.DataFrame(cm, index=CLASSES, columns=CLASSES)
    cm_df.to_csv(OUT_DIR / "confusion_matrix.csv")

    report_text = classification_report(oof_df["label"], oof_df["pred_label"], labels=CLASSES, zero_division=0)
    print(report_text)
    (REPORT_DIR / "classification_report.txt").write_text(report_text)

    print("Saved OOF and reports to:", OUT_DIR)
else:
    print("No OOF diagnostics generated.")



Loading checkpoint: /kaggle/working/outputs_final_kaggle/models/convnext_tiny_fb_in22k_ft_in1k_fold0_best.pth
  Model     : convnext_tiny.fb_in22k_ft_in1k
  Epoch     : 18
  Best acc  : 1.0


Predicting convnext_tiny_fb_in22k_ft_in1k_fold0_best.pth:   0%|          | 0/725 [00:00<?, ?it/s]

  Output shape: (23200, 29)

Loading checkpoint: /kaggle/working/outputs_final_kaggle/models/convnext_tiny_fb_in22k_ft_in1k_fold1_best.pth
  Model     : convnext_tiny.fb_in22k_ft_in1k
  Epoch     : 1
  Best acc  : 0.9996120689655172


Predicting convnext_tiny_fb_in22k_ft_in1k_fold1_best.pth:   0%|          | 0/725 [00:00<?, ?it/s]

  Output shape: (23200, 29)
OOF accuracy: 0.9998060344827586
              precision    recall  f1-score   support

           A       1.00      1.00      1.00      1600
           B       1.00      1.00      1.00      1600
           C       1.00      1.00      1.00      1600
           D       1.00      1.00      1.00      1600
           E       1.00      1.00      1.00      1600
           F       1.00      1.00      1.00      1600
           G       1.00      1.00      1.00      1600
           H       1.00      1.00      1.00      1600
           I       1.00      1.00      1.00      1600
           J       1.00      1.00      1.00      1600
           K       1.00      1.00      1.00      1600
           L       1.00      1.00      1.00      1600
           M       1.00      1.00      1.00      1600
           N       1.00      1.00      1.00      1600
           O       1.00      1.00      1.00      1600
           P       1.00      1.00      1.00      1600
           Q       1

In [16]:
# ============================================================
# 12) TEST INFERENCE + SUBMISSION.CSV
# ============================================================

trained_checkpoints = load_checkpoint_list()
print("Using checkpoints for ensemble:")
for c in trained_checkpoints:
    print(" -", c)

test_ds = SignDataset(test_df, transform=get_eval_tf(IMG_SIZE), has_labels=False)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

probs_sum = None
for ckpt_path in trained_checkpoints:
    probs = predict_checkpoint(ckpt_path, test_loader)
    if probs_sum is None:
        probs_sum = probs
    else:
        probs_sum += probs

avg_probs = probs_sum / len(trained_checkpoints)
pred_targets = avg_probs.argmax(axis=1)
pred_labels = [IDX_TO_CLASS[i] for i in pred_targets]

submission = test_df[["image_id"]].copy()
submission["label"] = pred_labels

# Strict checks before saving
assert list(submission.columns) == ["image_id", "label"]
assert len(submission) == len(test_df)
assert submission["image_id"].isna().sum() == 0
assert submission["label"].isna().sum() == 0
assert submission["image_id"].duplicated().sum() == 0
invalid_labels = sorted(set(submission["label"]) - set(CLASSES))
assert not invalid_labels, f"Invalid labels found: {invalid_labels}"

sub_path = SUBMISSION_DIR / "submission.csv"
submission.to_csv(sub_path, index=False)

# Also copy to /kaggle/working for easy download
submission.to_csv(WORKING_DIR / "submission.csv", index=False)

print("Saved submission:", sub_path)
print("Also saved:", WORKING_DIR / "submission.csv")
print("Rows:", len(submission))
print("Prediction distribution:")
print(submission["label"].value_counts().sort_index())
submission.head()


Using checkpoints for ensemble:
 - /kaggle/working/outputs_final_kaggle/models/convnext_tiny_fb_in22k_ft_in1k_fold0_best.pth
 - /kaggle/working/outputs_final_kaggle/models/convnext_tiny_fb_in22k_ft_in1k_fold1_best.pth

Loading checkpoint: /kaggle/working/outputs_final_kaggle/models/convnext_tiny_fb_in22k_ft_in1k_fold0_best.pth
  Model     : convnext_tiny.fb_in22k_ft_in1k
  Epoch     : 18
  Best acc  : 1.0


Predicting convnext_tiny_fb_in22k_ft_in1k_fold0_best.pth:   0%|          | 0/813 [00:00<?, ?it/s]

  Output shape: (26000, 29)

Loading checkpoint: /kaggle/working/outputs_final_kaggle/models/convnext_tiny_fb_in22k_ft_in1k_fold1_best.pth
  Model     : convnext_tiny.fb_in22k_ft_in1k
  Epoch     : 1
  Best acc  : 0.9996120689655172


Predicting convnext_tiny_fb_in22k_ft_in1k_fold1_best.pth:   0%|          | 0/813 [00:00<?, ?it/s]

  Output shape: (26000, 29)
Saved submission: /kaggle/working/outputs_final_kaggle/submissions/submission.csv
Also saved: /kaggle/working/submission.csv
Rows: 26000
Prediction distribution:
label
A          1846
B          1709
C           696
D            92
E          3238
F           111
G           962
H          1331
I           818
J          1044
K          1292
L          1281
M           142
N           199
O          1121
P          1228
Q           912
R           935
S           780
T           143
U          1153
V           643
W          1074
X           859
Y          1458
Z           692
del         209
nothing      15
space        17
Name: count, dtype: int64


,image_id,label
0,1.jpg,L
1,10.jpg,Q
2,100.jpg,W
3,1000.jpg,W
4,10000.jpg,G


In [17]:
# ============================================================
# 13) ZIP OUTPUTS FOR DOWNLOAD / REVIEW
# ============================================================

import shutil

zip_base = WORKING_DIR / "silent_gap_kaggle_outputs"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUT_DIR)
print("Created zip:", zip_path)

print("Important files:")
print(" -", WORKING_DIR / "submission.csv")
print(" -", OUT_DIR / "training_history.csv")
print(" -", OUT_DIR / "trained_checkpoints.csv")
print(" -", OUT_DIR / "confusion_matrix.csv")
print(" -", zip_path)


Created zip: /kaggle/working/silent_gap_kaggle_outputs.zip
Important files:
 - /kaggle/working/submission.csv
 - /kaggle/working/outputs_final_kaggle/training_history.csv
 - /kaggle/working/outputs_final_kaggle/trained_checkpoints.csv
 - /kaggle/working/outputs_final_kaggle/confusion_matrix.csv
 - /kaggle/working/silent_gap_kaggle_outputs.zip


## Notes for Reviewers

- This notebook uses **ConvNeXt Tiny** with transfer learning through `timm`.
- The default `RUN_MODE = "strong_3fold"` trains all 3 folds for 18 epochs (final submission run).
- For a quick sanity check, change `RUN_MODE` to `"quick"` in the config cell (1 fold, 5 epochs).
- The final `submission.csv` is saved to `/kaggle/working/submission.csv`.
- No Google Drive, Colab-specific mounting, or `kaggle.json` is required in this Kaggle version.
- **Internet must be enabled** in Kaggle settings for `timm` to download pretrained ConvNeXt weights.
- **GPU accelerator** (T4 or P100) is strongly recommended; CPU training will be extremely slow.
